In [1]:
# Cell 1: Mount Drive and Install Libraries
from google.colab import drive
drive.mount('/content/drive')

!pip install pymupdf marker-pdf accelerate

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.2/223.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.7/793.7 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

In [8]:
import fitz
import marker

print("PyMuPDF:", fitz.__doc__)
print("Marker imported successfully!")

PyMuPDF: PyMuPDF 1.28.0: Python bindings for the MuPDF 1.29.0 library.
Python 3.12 running on linux (64-bit).

Marker imported successfully!


In [9]:
# Cell 2: Split PDF by Chapters
import fitz  # PyMuPDF
import os
import re

# Define paths based on your Google Drive structure
base_dir = '/content/drive/MyDrive/UALink/'
input_pdf = os.path.join(base_dir, 'UALink200_Specification_v1.0_Evaluation_Copy.pdf')
chapters_dir = os.path.join(base_dir, 'Chapters')

# Create the output directory if it doesn't exist
os.makedirs(chapters_dir, exist_ok=True)

def sanitize_filename(name):
    return re.sub(r'[^\w\s\-_]', '', name).strip().replace(' ', '_')

# Open the PDF and extract the Table of Contents
doc = fitz.open(input_pdf)
toc = doc.get_toc()

chapters = []
for item in toc:
    level, title, page = item
    if level == 1:  # Target top-level bookmarks
        chapters.append({'title': title, 'start_page': page - 1})

if not chapters:
    print("No top-level bookmarks found. Double-check the PDF structure.")
else:
    # Determine end pages
    for i in range(len(chapters)):
        if i < len(chapters) - 1:
            chapters[i]['end_page'] = chapters[i + 1]['start_page'] - 1
        else:
            chapters[i]['end_page'] = len(doc) - 1

    print(f"Found {len(chapters)} main chapters. Splitting...")

    # Save each chapter to the Drive folder
    for idx, ch in enumerate(chapters, start=1):
        ch_title = sanitize_filename(ch['title'])
        output_filename = os.path.join(chapters_dir, f"{ch_title}.pdf")

        ch_doc = fitz.open()
        ch_doc.insert_pdf(doc, from_page=ch['start_page'], to_page=ch['end_page'])
        ch_doc.save(output_filename)
        ch_doc.close()
        print(f"Saved: {ch_title}.pdf")

doc.close()
print("Splitting complete! Check your Google Drive.")

Found 16 main chapters. Splitting...
Saved: Legal_Notice.pdf
Saved: Table_of_Contents.pdf
Saved: List_of_Figures.pdf
Saved: List_of_Tables.pdf
Saved: Revision_History.pdf
Saved: Preface.pdf
Saved: 1_Introduction.pdf
Saved: 2_UPLI_Interface_Definition_and_Operation_Rules.pdf
Saved: 3_Reliability_Availability_and_Serviceability_RAS.pdf
Saved: 4_UPLI_Interface_Reset_Signaling_and_Connection.pdf
Saved: 5_Transaction_Layer_TL.pdf
Saved: 6_Data_Link.pdf
Saved: 7_Physical_Layer.pdf
Saved: 8_Manageability_Requirements.pdf
Saved: 9_Security.pdf
Saved: 10__UALink_Switch_Requirements.pdf
Splitting complete! Check your Google Drive.


In [6]:
# # Cell 3: Convert All Chapters to Markdown (Batch Mode)
# import os
# import subprocess

# base_dir = '/content/drive/MyDrive/UALink/'
# chapters_dir = os.path.join(base_dir, 'Chapters')
# markdown_dir = os.path.join(base_dir, 'Markdown_Output')

# # Ensure the output directory exists
# os.makedirs(markdown_dir, exist_ok=True)

# print(f"Starting batch conversion of the entire folder: {chapters_dir}...")

# # CORRECTED: Use --output_dir flag for the destination
# command = [
#     "marker",
#     chapters_dir,
#     "--output_dir", markdown_dir,
#     "--output_format", "markdown"
# ]

# # Run the command
# result = subprocess.run(command, capture_output=True, text=True)

# if result.returncode == 0:
#     print("Batch conversion finished successfully! Check your Markdown_Output folder.")
#     # Optional: Print standard output to see Marker's internal logs
#     print(result.stdout)
# else:
#     print("Error during batch conversion:")
#     print(result.stderr)

Starting batch conversion of the entire folder: /content/drive/MyDrive/UALink/Chapters...


KeyboardInterrupt: 

In [10]:
# Cell 3: Convert Chapters to Markdown (Single File Mode)
import os
import subprocess

base_dir = '/content/drive/MyDrive/UALink/'
chapters_dir = os.path.join(base_dir, 'Chapters')
markdown_dir = os.path.join(base_dir, 'Markdown_Output')

os.makedirs(markdown_dir, exist_ok=True)

chapter_files = [f for f in os.listdir(chapters_dir) if f.endswith('.pdf')]
print(f"Starting conversion of {len(chapter_files)} files...")

for pdf_file in chapter_files:
    pdf_path = os.path.join(chapters_dir, pdf_file)
    print(f"Processing: {pdf_file}")

    # CORRECTED: Use marker_single and explicitly define output flags
    command = [
        "marker_single",
        pdf_path,
        "--output_dir", markdown_dir,
        "--output_format", "markdown"
    ]

    result = subprocess.run(command, capture_output=True, text=True)

    if result.returncode == 0:
        print(f"  -> Successfully converted {pdf_file}")
    else:
        print(f"  -> Error converting {pdf_file}:\n{result.stderr}")

print("\nAll conversions finished.")

Starting conversion of 16 files...
Processing: Legal_Notice.pdf
  -> Successfully converted Legal_Notice.pdf
Processing: Table_of_Contents.pdf
  -> Successfully converted Table_of_Contents.pdf
Processing: List_of_Figures.pdf
  -> Successfully converted List_of_Figures.pdf
Processing: List_of_Tables.pdf
  -> Successfully converted List_of_Tables.pdf
Processing: Revision_History.pdf
  -> Successfully converted Revision_History.pdf
Processing: Preface.pdf
  -> Successfully converted Preface.pdf
Processing: 1_Introduction.pdf
  -> Successfully converted 1_Introduction.pdf
Processing: 2_UPLI_Interface_Definition_and_Operation_Rules.pdf
  -> Successfully converted 2_UPLI_Interface_Definition_and_Operation_Rules.pdf
Processing: 3_Reliability_Availability_and_Serviceability_RAS.pdf
  -> Successfully converted 3_Reliability_Availability_and_Serviceability_RAS.pdf
Processing: 4_UPLI_Interface_Reset_Signaling_and_Connection.pdf
  -> Successfully converted 4_UPLI_Interface_Reset_Signaling_and_Conn

In [11]:
# Cell 4: Convert the Entire PDF to Markdown
import os
import subprocess

base_dir = '/content/drive/MyDrive/UALink/'
input_pdf = os.path.join(base_dir, 'UALink200_Specification_v1.0_Evaluation_Copy.pdf')
full_markdown_dir = os.path.join(base_dir, 'Markdown_Output_Full')

# Create a dedicated output directory for the full document
os.makedirs(full_markdown_dir, exist_ok=True)

print(f"Starting conversion of the full specification...")
print(f"File: {input_pdf}")
print("This may take a several minutes depending on the Colab GPU allocation.")

# Use marker_single for the original, uncut PDF
command = [
    "marker_single",
    input_pdf,
    "--output_dir", full_markdown_dir,
    "--output_format", "markdown"
]

# Run the command
result = subprocess.run(command, capture_output=True, text=True)

if result.returncode == 0:
    print("\nSuccessfully converted the full PDF!")
    print("Check the 'Markdown_Output_Full' folder in your Google Drive.")
else:
    print("\nError during conversion:")
    print(result.stderr)

Starting conversion of the full specification...
File: /content/drive/MyDrive/UALink/UALink200_Specification_v1.0_Evaluation_Copy.pdf
This may take a several minutes depending on the Colab GPU allocation.

Successfully converted the full PDF!
Check the 'Markdown_Output_Full' folder in your Google Drive.
